# 03_数据分析 (Data Analysis Agent)

**来源:** [LangChain Docs — Build a data analysis agent](https://docs.langchain.com/deep-agents/data-analysis)

本教程演示如何使用 Deep Agents 构建一个数据分析智能体。该智能体可以接收 CSV 文件、执行探索性数据分析、生成可视化图表，并通过 Slack 分享结果。数据分析和代码执行任务通常需要规划、编写脚本和处理产物（报告、图表），这正是 deep agents 的强项。

## 1. 核心概念

| 概念 | 说明 |
|------|------|
| 后端 (Backend) | 沙箱化代码执行环境，支持 Daytona / Modal / Runloop / AgentCore / LocalShell |
| 自定义工具 | 用于外部集成（如 Slack 消息发送） |
| 文件上传/下载 | 后端支持文件的上传、读写和下载 |
| Checkpointer | 用于支持多轮对话的持久化检查点 |

## 2. 环境准备

安装核心依赖和可选的 Slack SDK、沙箱后端。

In [ ]:
# 安装核心依赖
# pip install deepagents

# 可选：Slack SDK（用于分享结果）
# pip install slack-sdk

# 可选：沙箱环境（选择一个）
# pip install langchain-daytona       # Daytona
# pip install langchain-modal         # Modal
# pip install langchain-runloop       # Runloop
# pip install langchain-agentcore-codeinterpreter  # AgentCore

# 设置 API 密钥
import os

# os.environ["ANTHROPIC_API_KEY"] = "your-api-key"  # 模型
# os.environ["GOOGLE_API_KEY"] = "your-api-key"    # Gemini
# os.environ["TAVILY_API_KEY"] = "your-api-key"    # 搜索（如需）
# os.environ["SLACK_USER_TOKEN"] = "your-slack-token"  # Slack（可选）
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = "your-api-key"

print("环境准备完成")

## 3. 配置沙箱后端

   Deep Agents 使用后端在沙箱化环境中执行代码。支持多种沙箱提供商。选择一个适合你的环境。

In [ ]:
# --- Daytona 沙箱 ---
# from daytona import Daytona
# from langchain_daytona import DaytonaSandbox
# sandbox = Daytona().create()
# backend = DaytonaSandbox(sandbox=sandbox)
# result = backend.execute("echo ready")
# print(result)

# --- Modal 沙箱 ---
# import modal
# from langchain_modal import ModalSandbox
# app = modal.App.lookup("your-app")
# modal_sandbox = modal.Sandbox.create(app=app)
# backend = ModalSandbox(sandbox=modal_sandbox)

# --- Runloop 沙箱 ---
# from runloop_api_client import RunloopSDK
# from langchain_runloop import RunloopSandbox
# api_key = "..."
# client = RunloopSDK(bearer_token=api_key)
# devbox = client.devbox.create()
# backend = RunloopSandbox(devbox=devbox)

# --- AgentCore 沙箱 ---
# from bedrock_agentcore.tools.code_interpreter_client import CodeInterpreter
# from langchain_agentcore_codeinterpreter import AgentCoreSandbox
# interpreter = CodeInterpreter(region="us-west-2")
# interpreter.start()
# backend = AgentCoreSandbox(interpreter=interpreter)

# --- 本地 Shell 后端（仅用于开发测试） ---
# from deepagents.backends import LocalShellBackend
# backend = LocalShellBackend(root_dir=".", env={"PATH": "/usr/bin:/bin"})

print("后端配置就绪（取消注释并选择一个沙箱提供商）")

## 4. 上传示例数据

   创建示例销售数据并上传到后端，供智能体分析。

In [ ]:
import csv
import io

# 创建示例销售数据
data = [
    ["日期", "产品", "销量", "收入"],
    ["2025-08-01", "Widget A", 10, 250],
    ["2025-08-02", "Widget B", 5, 125],
    ["2025-08-03", "Widget A", 7, 175],
    ["2025-08-04", "Widget C", 3, 90],
    ["2025-08-05", "Widget B", 8, 200],
]

# 转换为 CSV 字节流
text_buf = io.StringIO()
writer = csv.writer(text_buf)
writer.writerows(data)
csv_bytes = text_buf.getvalue().encode("utf-8")
text_buf.close()

print("示例数据已准备就绪")
print(f"数据大小: {len(csv_bytes)} 字节")
print(f"包含 {len(data)-1} 条销售记录")

In [ ]:
# 将数据上传到后端
# 注意：需要先初始化有效的 backend

# backend.upload_files([("/home/daytona/data/sales_data.csv", csv_bytes)])
# print("数据已上传到后端")

# 也可以使用本地文件系统
import os
from pathlib import Path

local_data_dir = Path("./data_analysis_sample")
local_data_dir.mkdir(exist_ok=True)

with open(local_data_dir / "sales_data.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(data)

print(f"数据已写入 {local_data_dir / 'sales_data.csv'}")

## 5. 实现 Slack 工具（可选）

   数据分析任务可能产生产物（报告或图表）。以下工具从后端下载文件并通过 Slack SDK 发送。

In [ ]:
import os
from langchain.tools import tool


@tool(parse_docstring=True)
def slack_send_message(text: str, file_path: str | None = None) -> str:
    """发送消息，可选地包含附件（如图片）。

    参数:
        text: (str) 消息文本内容
        file_path: (str) 文件系统中附件的路径
    """
    from slack_sdk import WebClient

    slack_token = os.environ["SLACK_USER_TOKEN"]
    slack_client = WebClient(token=slack_token)

    if not file_path:
        slack_client.chat_postMessage(channel="C0123456ABC", text=text)
    else:
        # 从后端下载文件
        fp = backend.download_files([file_path])
        slack_client.files_upload_v2(
            channel="C0123456ABC",
            content=fp[0].content,
            initial_comment=text,
        )

    return "消息已发送"


print("Slack 工具定义完成")
print("注意：需要有效的 SLACK_USER_TOKEN 和 Slack 频道 ID")

## 6. 创建数据分析智能体

   实例化智能体，配置模型、工具、后端和检查点。

In [ ]:
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver
from deepagents import create_deep_agent

# 创建内存检查点（支持多轮对话）
checkpointer = InMemorySaver()

# 创建数据分析智能体
agent = create_deep_agent(
    # 选择模型
    # model="google_genai:gemini-3.5-flash",
    # model="anthropic:claude-sonnet-4-6",
    # model="openai:gpt-5.4",
    model="google_genai:gemini-3.5-flash",
    tools=[slack_send_message],  # 自定义工具
    backend=backend,             # 沙箱后端
    checkpointer=checkpointer,   # 对话检查点
)

# 生成线程 ID
thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}

print("数据分析智能体创建完成")
print(f"对话线程 ID: {thread_id}")

## 7. 运行数据分析任务

   向智能体提交数据分析请求，观察它如何探索数据、编写分析脚本、生成图表并分享结果。

In [ ]:
# 示例：运行数据分析任务
# 注意：需要配置有效的后端和 API 密钥

# input_message = {
#     "role": "user",
#     "content": (
#         "Analyze ./data/sales_data.csv in the current dir and generate a beautiful plot. "
#         "When finished, send your analysis and the plot to Slack using the tool."
#     ),
# }

# for step in agent.stream(
#     {"messages": [input_message]},
#     config,
#     stream_mode="updates",
# ):
#     for _, update in step.items():
#         if update and (messages := update.get("messages")):
#             for message in messages:
#                 message.pretty_print()

print("运行数据分析任务（取消注释以执行）")

## 8. 沙箱后端对比

    | 后端 | 安装命令 | 适用场景 | 安全等级 |
    |------|----------|----------|----------|
    | Daytona | `pip install langchain-daytona` | 通用沙箱 | 高 |
    | Modal | `pip install langchain-modal` | 云端执行 | 高 |
    | Runloop | `pip install langchain-runloop` | 托管沙箱 | 高 |
    | AgentCore | `pip install langchain-agentcore-codeinterpreter` | AWS 集成 | 高 |
    | LocalShell | 内置（无需安装） | 开发测试 | 低 |

    **安全提醒**: LocalShell 后端提供不受限制的文件系统和 Shell 访问，仅用于受控的开发环境。生产环境请使用沙箱后端。

## 9. 数据探索脚本示例

   以下是智能体可能生成的数据分析 Python 脚本的简化版本。

In [ ]:
# %%writefile analyze_sales.py
# 数据分析脚本示例 - 智能体可能会生成的代码

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# 设置图表样式
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11

# 读取数据
df = pd.read_csv('./data_analysis_sample/sales_data.csv')
df['日期'] = pd.to_datetime(df['日期'])

# 打印分析报告
print("=" * 60)
print("销售数据分析报告")
print("=" * 60)
print(f"\n1. 数据概览")
print(f"日期范围: {df['日期'].min().strftime('%Y-%m-%d')} 至 {df['日期'].max().strftime('%Y-%m-%d')}")
print(f"总记录数: {len(df)}")
print(f"产品种类: {', '.join(df['产品'].unique())}")

print(f"\n2. 汇总统计")
print(f"总收入: ${df['收入'].sum():,.2f}")
print(f"总销量: {df['销量'].sum()}")
print(f"平均日收入: ${df['收入'].mean():.2f}")
print(f"平均单笔销量: {df['销量'].mean():.2f}")

print(f"\n3. 产品表现")
product_stats = df.groupby('产品').agg({
    '收入': ['sum', 'mean'],
    '销量': ['sum', 'mean']
}).round(2)
print(product_stats)

print(f"\n4. 最佳销售日")
best_day = df.loc[df['收入'].idxmax()]
print(f"日期: {best_day['日期'].strftime('%Y-%m-%d')}")
print(f"产品: {best_day['产品']}")
print(f"收入: ${best_day['收入']:.2f}")
print(f"销量: {best_day['销量']}")

In [ ]:
# 创建可视化仪表板
fig = plt.figure(figsize=(16, 12))
fig.suptitle('销售数据分析仪表板', fontsize=20, fontweight='bold', y=0.995)

# 1. 每日收入
ax1 = plt.subplot(2, 3, 1)
colors = sns.color_palette("husl", len(df))
bars = ax1.bar(df['日期'].dt.strftime('%m-%d'), df['收入'], 
               color=colors, edgecolor='black', linewidth=1.5)
ax1.set_title('每日收入', fontsize=14, fontweight='bold', pad=10)
ax1.set_xlabel('日期')
ax1.set_ylabel('收入 ($)')
ax1.grid(axis='y', alpha=0.3)
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'${int(height)}', ha='center', va='bottom')

# 2. 每日销量
ax2 = plt.subplot(2, 3, 2)
bars = ax2.bar(df['日期'].dt.strftime('%m-%d'), df['销量'],
               color=colors, edgecolor='black', linewidth=1.5)
ax2.set_title('每日销量', fontsize=14, fontweight='bold', pad=10)
ax2.set_xlabel('日期')
ax2.set_ylabel('销量')
ax2.grid(axis='y', alpha=0.3)
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}', ha='center', va='bottom')

# 3. 各产品收入占比（饼图）
ax3 = plt.subplot(2, 3, 3)
product_revenue = df.groupby('产品')['收入'].sum()
wedges, texts, autotexts = ax3.pie(
    product_revenue, labels=product_revenue.index, autopct='%1.1f%%',
    colors=sns.color_palette("Set2", len(product_revenue)),
    startangle=90, explode=[0.05] * len(product_revenue))
ax3.set_title('各产品收入分布', fontsize=14, fontweight='bold', pad=10)

# 4. 各产品总收入（横向柱状图）
ax4 = plt.subplot(2, 3, 4)
product_revenue_sorted = product_revenue.sort_values(ascending=False)
bars = ax4.barh(product_revenue_sorted.index, product_revenue_sorted.values,
                color=sns.color_palette("coolwarm", len(product_revenue_sorted)),
                edgecolor='black', linewidth=1.5)
ax4.set_title('各产品总收入', fontsize=14, fontweight='bold', pad=10)
ax4.set_xlabel('收入 ($)')

# 5. 各产品总销量
ax5 = plt.subplot(2, 3, 5)
product_units = df.groupby('产品')['销量'].sum().sort_values(ascending=False)
bars = ax5.barh(product_units.index, product_units.values,
                color=sns.color_palette("viridis", len(product_units)),
                edgecolor='black', linewidth=1.5)
ax5.set_title('各产品总销量', fontsize=14, fontweight='bold', pad=10)
ax5.set_xlabel('销量')

# 6. 产品销售占比
ax6 = plt.subplot(2, 3, 6)
product_counts = df['产品'].value_counts()
wedges, texts, autotexts = ax6.pie(
    product_counts, labels=product_counts.index, autopct='%1.1f%%',
    colors=sns.color_palette("muted", len(product_counts)),
    startangle=45, explode=[0.05] * len(product_counts))
ax6.set_title('产品销售交易占比', fontsize=14, fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig('./data_analysis_sample/sales_analysis_plot.png', 
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("可视化仪表板已保存")

## 10. 关键设计原则

    | 原则 | 说明 |
    |------|------|
    | 凭证隔离 | 避免将凭证和密钥放入沙箱，使用工具层管理 |
    | 产物管理 | 分析任务产生的脚本、报告和图表通过 `download_files` 获取 |
    | 沙箱优先 | 生产环境始终使用沙箱后端，避免 LocalShell |
    | 检查点支持 | 使用 `InMemorySaver` 或持久化检查点支持多轮对话 |
    | 工具扩展 | Slack 工具可替换为其他渠道（邮件、Webhook 等） |

---

**参考链接**
- [LangChain Docs — Deep Agents Data Analysis](https://docs.langchain.com/deep-agents/data-analysis)
- [Deep Agents GitHub](https://github.com/langchain-ai/deep-agents)
- [Daytona Sandbox](https://github.com/daytonaio)
- [Modal Sandbox](https://modal.com)
- [Runloop Sandbox](https://runloop.ai)
- [Slack SDK for Python](https://slack.dev/python-slack-sdk/)